# SymboLM (v1.1) — Kaggle Stage 1 SFT Resume (Steps 200 → 675)
**Author:** Sanish Gyawali  
**Model:** `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`  
**Hardware:** NVIDIA Tesla T4 / P100 (16GB VRAM) in native FP16 (Unquantized LoRA)

This notebook resumes SFT training from `checkpoint-200` without any quantization mismatch, ensuring 100% numerical consistency with the TPU run.

In [ ]:
# 1. GPU Verification
!nvidia-smi

In [ ]:
# 2. Clone SymboLM and install dependencies
%cd /kaggle/working
!rm -rf symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git
%cd /kaggle/working/symboLM
!pip install -q -r requirements.txt

In [ ]:
# 3. Stage Checkpoint-200
# Check if attached via Kaggle Dataset or fallback to direct download
import os
import shutil
from pathlib import Path

dest_ckpt = Path("/kaggle/working/symboLM/checkpoints/sft_adapter/checkpoint-200")
dest_tok = Path("/kaggle/working/symboLM/checkpoints/tokenizer_extended")
dest_ckpt.mkdir(parents=True, exist_ok=True)
dest_tok.mkdir(parents=True, exist_ok=True)

# Search Kaggle input datasets
kaggle_inputs = list(Path("/kaggle/input").glob("**/*checkpoint-200*"))
if kaggle_inputs:
    src = kaggle_inputs[0] if kaggle_inputs[0].is_dir() else kaggle_inputs[0].parent
    print(f"Found checkpoint in Kaggle Dataset input: {src}")
    shutil.copytree(src, dest_ckpt, dirs_exist_ok=True)
    print("Successfully staged checkpoint-200!")
else:
    print("No Kaggle dataset input detected. Check next cell for direct Google Drive / HF download.")

In [ ]:
# (Option B) Direct download from Google Drive via gdown:
# 1. In Google Drive, right-click 'checkpoint-200' -> Share -> 'Anyone with the link can view'
# 2. Paste the folder URL below and run:
# !pip install -q gdown
# !gdown --folder 'PASTE_YOUR_GDRIVE_FOLDER_LINK_HERE' -O /kaggle/working/symboLM/checkpoints/sft_adapter/checkpoint-200

# (Option C) Download from HuggingFace Hub:
# !pip install -q huggingface_hub
# !huggingface-cli download gyawalisanish0/symboLM-sft-ckpt200 --local-dir /kaggle/working/symboLM/checkpoints/sft_adapter/checkpoint-200

In [ ]:
# 4. Resume SFT Training (Steps 200 -> 675)
# --no_quant uses native 16-bit LoRA with adamw_torch matching Checkpoint-200
!python -m training.sft_train \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --data_dir ./data/symbolic_dataset \
    --output_dir ./checkpoints/sft_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --lr 0.0002 \
    --no_quant \
    --resume_from_checkpoint ./checkpoints/sft_adapter/checkpoint-200

In [ ]:
# 5. Test Inference
!python -m inference.generate \
    --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
    --adapter_path ./checkpoints/sft_adapter \
    --prompt "A train departs with 100 passengers. At the first stop, 20 leave and 35 enter. How many passengers now?"

In [ ]:
# 6. Compress Final Trained Adapter for Easy Download from Kaggle Output
!tar -czvf /kaggle/working/symboLM_sft_final_adapter.tar.gz -C /kaggle/working/symboLM/checkpoints/sft_adapter .
print("Done! Output is saved to /kaggle/working/symboLM_sft_final_adapter.tar.gz")